In [1]:
from pathlib import Path
import sys

# Resolve paths even if the working directory is repo root.
notebook_dir = Path.cwd()
if not (notebook_dir / "conf.toml").exists():
    notebook_dir = notebook_dir / "notebook"
repo_root = notebook_dir.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

paths = {
    "conf": notebook_dir / "conf.toml",
    "doc": notebook_dir / "blob_documentation.txt",
    "code": notebook_dir / "blob.js",
    "spec": notebook_dir / "blob.webidl",
}

In [2]:
from circinus.settings import load_config

# Circinus demo overview
This notebook runs a small, end-to-end demo of the generation-based fuzzer. It sets up paths, loads config, creates an `Agent`, then fuzzes either the bundled WebIDL artifacts or the small demo files below.

**Note:** Circinus now uses OpenAI-compatible model calls with a local Ollama-first default. Set the `LLM_*` environment variables or edit `notebook/conf.toml` before running cells that invoke the LLM.

In [3]:
import json

In [4]:
config = load_config(str(paths["conf"]))
print(f"Config file: {paths['conf']}")
print(f"LLM model: {config.llm_model}")
print(f"LLM base URL: {config.llm_base_url}")

Config file: c:\Users\alvar\OneDrive\Documents\Custom Office Templates\Desktop\Github\Senior Project\LLMFuzzer\notebook\conf.toml
LLM model: qwen3
LLM base URL: http://localhost:11434/v1


In [5]:
artifact_paths = {
    "report": repo_root / "demo" / "shot-vuln.md",
    "cache_file": repo_root / "demo" / "shot-seed-cache.json",
    "cache_replay": repo_root / "demo" / "shot-cache-replay",
    "afl_dir": repo_root / "demo" / "shot-afl",
}

for label, path in artifact_paths.items():
    print(f"{label}: {path}")

report: c:\Users\alvar\OneDrive\Documents\Custom Office Templates\Desktop\Github\Senior Project\LLMFuzzer\demo\shot-vuln.md
cache_file: c:\Users\alvar\OneDrive\Documents\Custom Office Templates\Desktop\Github\Senior Project\LLMFuzzer\demo\shot-seed-cache.json
cache_replay: c:\Users\alvar\OneDrive\Documents\Custom Office Templates\Desktop\Github\Senior Project\LLMFuzzer\demo\shot-cache-replay
afl_dir: c:\Users\alvar\OneDrive\Documents\Custom Office Templates\Desktop\Github\Senior Project\LLMFuzzer\demo\shot-afl


In [6]:
print("This notebook shows three Circinus contributions in order.")
print("1) Crash reporter -> plain-English vulnerability summary")
print("2) Seed cache -> saved seeds replayed on the next run")
print("3) AFL++ handoff -> generated seeds fed into external fuzzing")

This notebook shows three Circinus contributions in order.
1) Crash reporter -> plain-English vulnerability summary
2) Seed cache -> saved seeds replayed on the next run
3) AFL++ handoff -> generated seeds fed into external fuzzing


## 1. Automated vulnerability reporter
Show the report sample first so the audience sees the output Circinus creates from crash artifacts. The key point is that raw crash data becomes a readable summary with root cause, reproducibility, and remediation guidance.

In [7]:
report_text = artifact_paths["report"].read_text(encoding="utf-8")
print(report_text)

# Circinus Vulnerability Report

- target: demo\shot-seeds\CRASH.txt
- generated_utc: 2026-04-28T04:51:09.630363+00:00

Probable root cause: the demo target exits with a RuntimeError when the input contains the token CRASH.

Exploitability: low in this toy target, but it proves crash detection and report generation.

Reproducibility: run the target against the same file and the exception is immediate.

Remediation: tighten input validation and remove crash-triggering sentinel paths from production code.



## 2. Seed cache replay
Next, show the cache replay folder so the audience can see that Circinus keeps crash-derived seeds and writes them back into a new run. That is the persistence story: useful inputs are not lost after one run finishes.

In [10]:
cache_data = json.loads(artifact_paths["cache_file"].read_text(encoding="utf-8"))
replayed_files = sorted(path.name for path in artifact_paths["cache_replay"].glob("cached-*.txt"))

print(f"Cache version: {cache_data.get('version')}")
print(f"Seed count: {len(cache_data.get('seeds', []))}")
print("Hydrated files:")
for name in replayed_files:
    print(f"- {name}")

Cache version: 1
Seed count: 2
Hydrated files:
- cached-0000.txt
- cached-0001.txt


## 3. AFL++ handoff
Finish with the AFL++ output directory. This is the integration story: Circinus produces a starting corpus, AFL++ picks it up, and the run output shows the external fuzzing job moving forward.

In [11]:
afl_dir = artifact_paths["afl_dir"]
cmdline = (afl_dir / "cmdline").read_text(encoding="utf-8").strip()
fuzzer_setup = (afl_dir / "fuzzer_setup").read_text(encoding="utf-8").strip()
plot_data = (afl_dir / "plot_data").read_text(encoding="utf-8").strip().splitlines()
queue_files = sorted(path.name for path in (afl_dir / "queue").glob("id*"))

print("AFL++ command line:")
print(cmdline)
print()
print("AFL++ setup:")
print(fuzzer_setup)
print()
print("Plot data:")
for line in plot_data:
    print(line)
print()
print(f"Queue entries: {len(queue_files)}")
for name in queue_files:
    print(f"- {name}")

AFL++ command line:
python3
demo/target.py
@@

AFL++ setup:
# environment variables:
AFL_CUSTOM_INFO_PROGRAM=python3
AFL_CUSTOM_INFO_PROGRAM_ARGV=demo/target.py @@
AFL_CUSTOM_INFO_OUT=demo/shot-afl
AFL_I_DONT_CARE_ABOUT_MISSING_CRASHES=1
AFL_SKIP_BIN_CHECK=1
# command line:
'afl-fuzz' '-n' '-V' '5' '-i' 'demo/shot-seeds' '-o' 'demo/shot-afl' '--' 'python3' 'demo/target.py' '@@'

Plot data:
# relative_time, cycles_done, cur_item, corpus_count, pending_total, pending_favs, map_size, saved_crashes, saved_hangs, max_depth, execs_per_sec, total_execs, edges_found
5, 0, 2, 3, 1, 0, 0.00%, 0, 0, 1, 28.88, 169, 0

Queue entries: 3
- id000000,time0,execs0,orig0-1.txt
- id000001,time0,execs0,orig0-2.txt
- id000002,time0,execs0,origCRASH.txt


## Summed Up

1. the report proves Circinus can turn a crash into a readable finding,
2. the cache shows seeds survive between runs,
3. the AFL++ output shows Circinus handing off work to another fuzzer.

That gives you a simple story arc: detect, preserve, and scale the fuzzing work.